# Notebook 04 — Features Lingüísticas Interpretables

## Objetivo

Extraer un conjunto de **features lingüísticas explícitas** por artículo y entrenar un
clasificador de **Regresión Logística** sobre ellas (Experimento 2 del informe).

A diferencia de los modelos BoW/TF-IDF del Notebook 03, aquí el objetivo **no es maximizar
performance** sino la **interpretabilidad de coeficientes**: un coeficiente alto en
`ratio_exclamacion` se lee directamente como "más exclamaciones → más probable fake".

La extracción vive en `src/nlp/features.py` y se cachea en `data/processed/`.
Referencia metodológica: [ADR Experimento 2](../docs/adr/experimento-02-features-linguisticas.md).

In [1]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from nlp.features import (
    FEATURE_NAMES,
    TEXT_FIELDS,
    coefficients_dataframe,
    evaluate_linguistic_model,
    load_or_extract_features,
    load_source_normalization_decision,
    select_best_text_field,
    tune_logistic_regression,
)
from nlp.io import load_splits
from nlp.metrics import consolidate_results, metrics_to_row
from nlp.paths import (
    DEV_MODE,
    RESULTS_FIGURES,
    RESULTS_METRICS,
    SOURCE_ABLATION_DECISION,
)
from nlp.plotting import save_figure, setup_style

setup_style()
FIGURE_PREFIX = "04_"

LINGUISTIC_COLS = ["label", "title_text", "body_text", "full_text"]
DATASET_SCOPE = "politics"

## 1. Carga de splits (subconjunto político)

In [2]:
train, val, test = load_splits(DATASET_SCOPE, columns=LINGUISTIC_COLS)

if DEV_MODE:
    sample_frac = 0.1
    train = train.sample(frac=sample_frac, random_state=42).reset_index(drop=True)
    val = val.sample(frac=sample_frac, random_state=42).reset_index(drop=True)
    test = test.sample(frac=sample_frac, random_state=42).reset_index(drop=True)
    print(f"DEV_MODE activo: muestra al {sample_frac:.0%} por split")

train.shape, val.shape, test.shape

((12635, 4), (2707, 4), (2708, 4))

## 2. Decisión de ablación de fuente (Experimento 1)

In [3]:
source_decision = load_source_normalization_decision(SOURCE_ABLATION_DECISION)
normalize_source = source_decision["use_source_normalization"]

if normalize_source:
    print(
        f"Normalizar fuentes en extracción "
        f"(caída F2 val = {source_decision['f2_drop']:.3f} "
        f"≥ {source_decision['threshold']})."
    )
else:
    print(
        f"Texto original en extracción "
        f"(caída F2 val = {source_decision['f2_drop']:.3f} "
        f"< {source_decision['threshold']})."
    )

Texto original en extracción (caída F2 val = 0.020 < 0.03).


## 3. Extracción de features lingüísticas

Vector de **8 features explícitas** por artículo sobre texto crudo (`title_text`,
`body_text`, `full_text`). Sin `StandardScaler`: los coeficientes del LR quedan en
unidades originales.

| Feature | Herramienta | Definición operativa |
| :------ | :---------- | :------------------- |
| `ratio_exclamacion` | regex | `!` / nº de oraciones |
| `ratio_mayusculas` | regex | palabras ALL CAPS (len>1) / palabras |
| `long_oracion_prom` | spaCy | tokens medios por oración |
| `ratio_adj_sust` | spaCy POS | ADJ / (NOUN + PROPN) |
| `sentimiento_vader` | VADER | score compuesto |
| `densidad_ner` | spaCy NER | entidades / oración |
| `freq_url` | conteo | ocurrencias de `[URL]` |
| `freq_pronombres` | spaCy morph | PRON 1.ª/2.ª persona / tokens |

In [4]:
nlp = spacy.load("en_core_web_sm")
vader = SentimentIntensityAnalyzer()

splits = {"train": train, "val": val, "test": test}
feature_store: dict[str, dict[str, pd.DataFrame]] = {}

for field in TEXT_FIELDS:
    feature_store[field] = {}
    for split_name, split_df in splits.items():
        feature_store[field][split_name] = load_or_extract_features(
            split_df,
            field,
            DATASET_SCOPE,
            split_name,
            normalize_source=normalize_source,
            nlp=nlp,
            vader=vader,
        )

feature_store["full_text"]["train"].head()

Features lingüísticas:   0%|          | 0/12635 [00:00<?, ?it/s]

Features lingüísticas:   0%|          | 0/2707 [00:00<?, ?it/s]

Features lingüísticas:   0%|          | 0/2708 [00:00<?, ?it/s]

Features lingüísticas:   0%|          | 0/12635 [00:00<?, ?it/s]

Features lingüísticas:   0%|          | 0/2707 [00:00<?, ?it/s]

Features lingüísticas:   0%|          | 0/2708 [00:00<?, ?it/s]

Features lingüísticas:   0%|          | 0/12635 [00:00<?, ?it/s]

Features lingüísticas:   0%|          | 0/2707 [00:00<?, ?it/s]

Features lingüísticas:   0%|          | 0/2708 [00:00<?, ?it/s]

,ratio_exclamacion,ratio_mayusculas,long_oracion_prom,ratio_adj_sust,sentimiento_vader,densidad_ner,freq_url,freq_pronombres
0,0.00,0.026946,20.421053,0.261261,0.9732,1.842105,0.0,0.002577
1,0.00,0.006014,34.022222,0.043029,-0.9969,4.311111,0.0,0.003266
2,0.00,0.051948,20.142857,0.341176,-0.9355,1.928571,0.0,0.021277
3,0.00,0.050682,32.722222,0.211765,-0.3736,2.944444,0.0,0.005093
4,0.05,0.021077,24.050000,0.184397,0.9905,2.650000,0.0,0.004158


## 4. Sub-experimento — título vs. cuerpo vs. combinado (Hipótesis 2)

Regresión Logística sin scaler; selección de `C ∈ {0.1, 1, 10}` por **F2 fake** en validación.

In [5]:
field_results = []
fitted_models = {}

for field in TEXT_FIELDS:
    X_train = feature_store[field]["train"]
    X_val = feature_store[field]["val"]
    clf, best_c, val_metrics = tune_logistic_regression(
        X_train,
        train["label"],
        X_val,
        val["label"],
    )
    fitted_models[field] = clf
    field_results.append(
        {
            **val_metrics,
            "text_field": field,
            "best_param": best_c,
            "split": "val",
        }
    )

field_comparison = pd.DataFrame(field_results).sort_values(
    "f2_fake", ascending=False
)
field_comparison[["text_field", "f2_fake", "recall_fake", "precision_fake", "best_param"]]

,text_field,f2_fake,recall_fake,precision_fake,best_param
0,title_text,0.917677,0.909827,0.950483,10.0
2,full_text,0.833528,0.825434,0.867558,10.0
1,body_text,0.766101,0.761850,0.783591,10.0


In [6]:
# Hipótesis 2: F2 por campo de texto (título vs cuerpo vs combinado).
# Visualiza que el TÍTULO supera al cuerpo y al combinado → H2 refutada.
fig, ax = plt.subplots(figsize=(7, 4))
order = field_comparison.sort_values("f2_fake", ascending=False)
sns.barplot(
    data=order, x="text_field", y="f2_fake", hue="text_field",
    legend=False, palette="viridis", ax=ax,
)
for i, v in enumerate(order["f2_fake"]):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center")
ax.set_ylim(0, 1.05)
ax.set_ylabel("F2 fake (validación)")
ax.set_xlabel("Campo de texto")
ax.set_title("Hipótesis 2: F2 por campo — el título supera al cuerpo y al combinado")
save_figure(fig, RESULTS_FIGURES / f"{FIGURE_PREFIX}h2_f2_by_text_field.png")
plt.show()

## 5. Coeficientes por feature (interpretabilidad)

Positivo → asociado a fake; negativo → asociado a real. Se muestran para el campo con
mejor F2 en validación.

In [7]:
best_field = select_best_text_field(field_comparison)
best_clf = fitted_models[best_field]
coef_df = coefficients_dataframe(best_clf, FEATURE_NAMES)
print(f"Mejor campo en val: {best_field}")
coef_df

Mejor campo en val: title_text


,feature,coefficient,abs_coefficient
0,ratio_mayusculas,29.612523,29.612523
1,ratio_exclamacion,13.746884,13.746884
2,freq_pronombres,11.883090,11.883090
3,ratio_adj_sust,-3.274684,3.274684
4,densidad_ner,-1.463103,1.463103
5,long_oracion_prom,0.390739,0.390739
6,sentimiento_vader,-0.365922,0.365922
7,freq_url,0.000000,0.000000


In [8]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = coef_df.sort_values("coefficient")
colors = ["#e74c3c" if c > 0 else "#2ecc71" for c in plot_df["coefficient"]]
ax.barh(plot_df["feature"], plot_df["coefficient"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Coeficiente LR (sin scaler)")
ax.set_title(f"Coeficientes — {best_field}")
save_figure(fig, RESULTS_FIGURES / f"{FIGURE_PREFIX}linguistic_coefficients.png")
plt.show()

## 6. Evaluación final en test y persistencia

Persistir la **tabla comparativa de F2 en validación** para las tres condiciones de campo
(título/cuerpo/combinado, Hipótesis 2) y evaluar la mejor configuración **una sola vez** en
test. `consolidate_results` toma únicamente la fila de test para la tabla unificada.

In [9]:
result_rows = []

# Tabla comparativa de F2 en validación para las tres condiciones de campo
# (Hipótesis 2): se persisten título/cuerpo/combinado, no solo el ganador.
for field in TEXT_FIELDS:
    field_c = float(
        field_comparison.loc[
            field_comparison["text_field"] == field, "best_param"
        ].iloc[0]
    )
    metrics, _, _ = evaluate_linguistic_model(
        fitted_models[field], feature_store[field]["val"], val["label"]
    )
    result_rows.append(
        metrics_to_row(
            metrics,
            {
                "model": "linguistic_features_lr",
                "dataset_scope": DATASET_SCOPE,
                "text_field": field,
                "best_param": field_c,
                "use_source_normalization": normalize_source,
                "split": "val",
            },
        )
    )

# Evaluación final en test: solo la mejor configuración, una única vez.
best_c = float(
    field_comparison.loc[
        field_comparison["text_field"] == best_field, "best_param"
    ].iloc[0]
)
test_metrics, _, _ = evaluate_linguistic_model(
    best_clf, feature_store[best_field]["test"], test["label"]
)
result_rows.append(
    metrics_to_row(
        test_metrics,
        {
            "model": "linguistic_features_lr",
            "dataset_scope": DATASET_SCOPE,
            "text_field": best_field,
            "best_param": best_c,
            "use_source_normalization": normalize_source,
            "split": "test",
        },
    )
)

results_df = pd.DataFrame(result_rows)
out_path = RESULTS_METRICS / "linguistic_features_results.csv"
results_df.to_csv(out_path, index=False)
print(f"Resultados guardados en {out_path}")
results_df

Resultados guardados en /Users/lucasrouco/workspace/ITBA/NLP/nlp/results/metrics/linguistic_features_results.csv


,accuracy,precision_fake,recall_fake,f1_fake,f2_fake,roc_auc,model,dataset_scope,text_field,best_param,use_source_normalization,split
0,0.956040,0.950483,0.909827,0.929711,0.917677,0.986073,linguistic_features_lr,politics,title_text,10.0,False,val
1,0.856668,0.783591,0.761850,0.772567,0.766101,0.888001,linguistic_features_lr,politics,body_text,10.0,False,val
2,0.903953,0.867558,0.825434,0.845972,0.833528,0.943493,linguistic_features_lr,politics,full_text,10.0,False,val
3,0.950148,0.956005,0.891353,0.922547,0.903574,0.980430,linguistic_features_lr,politics,title_text,10.0,False,test


In [10]:
all_results = consolidate_results()
linguistic_test = all_results[
    all_results["model"] == "linguistic_features_lr"
]
linguistic_test[["model", "text_field", "f2_fake", "recall_fake", "split"]]

,model,text_field,f2_fake,recall_fake,split
1,linguistic_features_lr,title_text,0.903574,0.891353,test


## Conclusiones

1. **Features de mayor magnitud (campo `title_text`, el de mejor F2 en val):** dominan tres features con signo positivo (→ fake): `ratio_mayusculas` (+29.6), `ratio_exclamacion` (+13.7) y `freq_pronombres` (+11.9). Con signo negativo (→ real): `ratio_adj_sust` (−3.27) y `densidad_ner` (−1.46). `long_oracion_prom` (+0.39) y `sentimiento_vader` (−0.37) aportan poco. `freq_url` es **0.0** en el título: los titulares casi no contienen `[URL]`, por lo que esta feature no contribuye al mejor modelo (sí tiene señal en cuerpo/combinado, donde viven los enlaces).

2. **Competitividad vs. baseline:** el LR de 8 features alcanza **F2 test = 0.904** (val 0.918), por **debajo** del baseline BoW/TF-IDF del Notebook 03 (~0.989). No es competitivo en sentido estricto, pero es notable que 8 variables interpretables capturen la mayor parte de la separación: el resultado lingüístico central del trabajo es *qué* features discriminan, no superar al baseline.

3. **Campo con más señal:** el **título** (val F2 0.918) supera ampliamente al cuerpo (0.766) y al combinado (0.834). La señal de estas 8 features se concentra en el titular.

4. **Hipótesis 2 — REFUTADA.** H2 predecía que el *cuerpo completo* clasificaría mejor que el *título solo*; los datos muestran lo contrario (título > combinado > cuerpo). En este corpus las features de estilo (mayúsculas, exclamaciones, pronombres de 1.ª/2.ª persona) son más densas y discriminativas en los titulares —típicamente sensacionalistas en las fake— que diluidas en el cuerpo. Los signos **confirman** las hipótesis del EDA sobre exclamaciones/mayúsculas (asociadas a fake) y sobre densidad de entidades (NER alta → estilo Reuters → real).

> Nota de validez: estos coeficientes describen el estilo *fake vs. Reuters* de este corpus político, no una ley general de la desinformación (la clase real proviene de una única fuente). Ver la sección de limitaciones del Informe.